In [2]:
#	project1_research_agent.py
from	typing	import	Literal
from	langgraph.graph	import	StateGraph,	MessagesState,	START,	END
from	langgraph.prebuilt	import	ToolNode
from	langgraph.checkpoint.memory	import	InMemorySaver
from	langchain.chat_models	import	init_chat_model
from	langchain_core.tools	import	tool

c:\Users\geetikak\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tools

In [3]:
from langchain_tavily import TavilySearch

search_web = TavilySearch(
    max_results=5,
    topic="general"
)


In [4]:
@tool
def calculator(expression: str) -> str:
    """Evaluate basic arithmetic expression"""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"
    
tools = [search_web, calculator]
    

In [7]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b")
SYSTEM_PROMPT	=	(
				"You	are	a	precise	research	assistant.	Use	search_web	for	facts	you	"
				"don't	know,	calculator	for	any	arithmetic.	Cite	which	tool	gave	you	"
				"an	answer.	Be	concise."
)
llm_with_tool = llm.bind_tools(tools=tools)

graph

In [8]:
def call_model(state: MessagesState) -> dict:
    messages = state['messages']
    if not messages or messages[0].type!='system':
        messages=[{"role":"system",	"content": SYSTEM_PROMPT}]+messages
    response=llm_with_tools.invoke(messages)
    return	{"messages":	[response]}

In [9]:
def	should_continue(state:	MessagesState)	->	Literal["tools",	"__end__"]:
				return	"tools"	if	state["messages"][-1].tool_calls	else	"__end__"

In [10]:
builder	=	StateGraph(MessagesState)
builder.add_node("agent",	call_model)
builder.add_node("tools",	ToolNode(tools))
builder.add_edge(START,	"agent")
builder.add_conditional_edges("agent",	should_continue)
builder.add_edge("tools",	"agent")
graph	=	builder.compile(checkpointer=InMemorySaver())

Run	with	memory	+	streaming

In [11]:
def	chat(thread_id:	str):
				config	=	{"configurable":	{"thread_id":	thread_id}}
				print("Research	Assistant	ready.	Type	'quit'	to	exit.\n")
				while	True:
								user_input	=	input("You:	")
								if	user_input.lower()	in	{"quit",	"exit"}:
												break
								print("Assistant:	",	end="",	flush=True)
								for	mode,	chunk	in	graph.stream(
												{"messages":	[{"role":	"user",	"content":	user_input}]},
												config,
												stream_mode=["messages",	"updates"],
								):
												if	mode	==	"messages":
																token,	meta	=	chunk
																if	meta.get("langgraph_node")	==	"agent"	and	token.content:
																				print(token.content,	end="",	flush=True)
								print("\n")

In [ ]:
if	__name__	==	"__main__":
				chat(thread_id="demo-session-1")


Research	Assistant	ready.	Type	'quit'	to	exit.

